# 自然语言处理：中文入门

**学习目标：**区分常见的 NLP 任务，解释分词与语言歧义，定义一个二分类情感分析实验。

| 任务 | 中文例子 | 输出 |
|---|---|---|
| 情感分析 | “这部电影意外地好看。” | 正面 |
| 命名实体识别 | “小王去了上海。” | 人名、地名 |
| 语义相似度 | “好看的片子” / “出色的电影” | 相似度分数 |
| 问答 | “导演是谁？” | 一段答案文本 |
| 文本生成 | “写一条短评。” | 生成的文本 |


In [1]:
import sys
from importlib.metadata import version
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
assert sys.version_info[:2] == (3, 12), "请选择项目的 Python 3.12 内核。"
assert version('paddlepaddle').split("+")[0] == '3.3.1', "请选择项目锁定的运行环境。"
print("Python：", sys.version.split()[0])
print('paddlepaddle' + "：", version('paddlepaddle'))
print("运行模式：", '课堂默认训练参数')

Python： 3.12.12
paddlepaddle： 3.3.1
运行模式： 课堂默认训练参数


**预期输出：**

```text
Python： 3.12.12
paddlepaddle： 3.3.1
运行模式： 课堂默认训练参数
```

## 语言为什么难

“不差”不等于“差”；“又是晚点，真棒”可能是反话；“银行关门了”需要上下文才能确定是河岸还是银行。
语序、否定词、领域和文本长度都会改变含义。正面/负面只是数据集给出的标签，
不能完整描述真实评论，褒贬混合和中立的表达依然存在。

中文不像英文那样天然用空格分词，因此中文课程使用 jieba 做词级切分，
并且不使用停用词表——“不”“没”这类否定词一旦删掉，情感就会反过来。
真实的工业系统更多使用预训练模型自带的子词切分器，本节先用最直白的方式建立直觉。


In [2]:
from pathlib import Path
import os
import sys

# 不依赖当前工作目录，向上找到中文课程根目录。
start = Path.cwd().resolve()
COURSE = next((p for p in [start, *start.parents]
               if (p / "CHINESE_COURSE_ROOT").is_file()), None)
if COURSE is None:
    raise RuntimeError("请在项目根目录（中文课程）中启动 Jupyter。")
ENGLISH = COURSE / "english"
# 中文专用模块在仓库根 common/，与英文课程共享的数据与检测模块在 english/common。
sys.path.insert(0, str(ENGLISH / "common"))
sys.path.insert(0, str(COURSE / "common"))
# 模型权重缓存与英文课程共用一份，避免同一份权重重复下载。
os.environ.setdefault("HF_HOME", str(ENGLISH / "data" / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(ENGLISH / "data" / "torch-cache"))

'/Users/will/workspace/deep-learning-lab/english/data/torch-cache'

**预期输出：**

```text
'…/deep-learning-lab/english/data/torch-cache'
```

（输出里的 `…` 代表运行本 notebook 时所在机器的仓库绝对路径前缀，你在自己机器上执行会看到自己的路径。）

In [3]:
from sentiment_cn import tokenize
examples = ["我很喜欢这段表演，但结局令人失望。",
            "这部电影一点也不差。", "真棒，又是一个能猜到的结局。"]
for text in examples:
    print("原文：", text)
    print("分词：", tokenize(text))

原文： 我很喜欢这段表演，但结局令人失望。


/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


分词： ['我', '很', '喜欢', '这', '段', '表演', '但', '结局', '令人', '失望']
原文： 这部电影一点也不差。
分词： ['这部', '电影', '一点', '也', '不', '差']
原文： 真棒，又是一个能猜到的结局。
分词： ['真棒', '又', '是', '一个', '能', '猜', '到', '的', '结局']


**预期输出：**

```text
原文： 我很喜欢这段表演，但结局令人失望。
/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
分词： ['我', '很', '喜欢', '这', '段', '表演', '但', '结局', '令人', '失望']
原文： 这部电影一点也不差。
分词： ['这部', '电影', '一点', '也', '不', '差']
原文： 真棒，又是一个能猜到的结局。
分词： ['真棒', '又', '是', '一个', '能', '猜', '到', '的', '结局']
```

## 本课程的实验

我们使用 [ChnSentiCorp 中文酒店评论数据集](https://github.com/pengchuanzhang/Senta)。
标签是 `负面=0`、`正面=1`。官方 `train.tsv` 按比例划出训练集与验证集；本发行包里的
`test.tsv` 只有 `qid` 和文本、没有标签（排行榜用的隐藏测试集），所以最终评估使用带标签的
官方 `dev.tsv`。预处理会剔除不含中文的记录并跨划分去掉完全重复的文本，
因此处理后的条数与论文公布的条数不同，口径写在 `provenance.json` 里。

词表构建和 Word2Vec 训练只用训练集文本；验证集和测试集中没见过的词映射到未知符号。
预训练情感模型是一份独立的可选对照，不是本节要训练的 LSTM。

**讨论：**褒贬混合的评论你会怎么打标签？为什么否定词必须留在输入里？
为什么课堂子集上的准确率不能当作基准成绩？
